# FYP Continual Learning Full Run (Colab)

This notebook runs the full training pipeline on Google Colab (T4-friendly).
It assumes your project and datasets are stored in Google Drive under `MyDrive`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Paths

Set your project root and data folder in Drive.
Expected CSV files: `demand_forecasting.csv` and `rl_environment.csv`.

In [ ]:
import sys
from pathlib import Path

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/fyp')
DATA_DIR = DRIVE_PROJECT_DIR / 'data' / 'processed'
OUTPUT_DIR = DRIVE_PROJECT_DIR / 'outputs' / 'colab_run_001'

assert DRIVE_PROJECT_DIR.exists(), f'Missing project folder: {DRIVE_PROJECT_DIR}'
assert DATA_DIR.exists(), f'Missing data folder: {DATA_DIR}'

sys.path.insert(0, str(DRIVE_PROJECT_DIR))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import subprocess
import sys

requirements_path = str(DRIVE_PROJECT_DIR / 'requirements.txt')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', requirements_path])

In [ ]:
import torch

from fyp_pipeline.core_pipeline import CONFIG, configure_vast_ai, prepare_data, print_task_summary
from fyp_pipeline.experiment_runner import (
    run_experiment,
    build_cl_summary,
    print_and_save_comparison_tables,
    generate_all_plots,
)

# Colab T4-friendly overrides
if torch.cuda.is_available():
    CONFIG['hardware']['precision'] = '16-mixed'
    CONFIG['hardware']['device'] = 'cuda'
    CONFIG['hardware']['pin_memory'] = True
else:
    CONFIG['hardware']['precision'] = '32'
    CONFIG['hardware']['device'] = 'cpu'
    CONFIG['hardware']['pin_memory'] = False

CONFIG['hardware']['num_workers'] = 2
CONFIG['hardware']['persistent_workers'] = False
CONFIG['forecasting']['batch_size'] = 128

configure_vast_ai(
    data_dir=str(DATA_DIR),
    output_dir=str(OUTPUT_DIR),
    require_gpu=False,
)

tft_tasks, rl_tasks, tft_df, rl_df = prepare_data()
print_task_summary(tft_tasks, rl_tasks)

In [ ]:
run_experiment(tft_tasks, rl_tasks)

In [ ]:
cl_summary = build_cl_summary()
tables = print_and_save_comparison_tables(cl_summary)

In [ ]:
generate_all_plots(cl_summary)

## Outputs

Results, checkpoints, logs, and plots are saved under:

- `/content/drive/MyDrive/fyp/outputs/colab_run_001`